# Deriving Feynman Rules with Symbolica

This notebook implements the canonical-quantization algorithm for extracting
Feynman vertex factors from a Lagrangian interaction term

As a reference i am following pag 57 of the original paper

**Summary Algorithm (canonical quantization)**

1. Select an interaction term $\mathcal{L}_{\text{int}} = g_{\alpha_1\dots\alpha_n}\,\partial\dots\partial\,\phi_{\alpha_1}\dots\phi_{\alpha_n}$
2. Attach creation operators $a^\dagger_{\alpha_n}(p_n)\dots a^\dagger_{\alpha_1}(p_1)$
3. Form the vacuum matrix element $\langle 0|\phi_{\alpha_1}(x)\dots\phi_{\alpha_n}(x)\,a^\dagger_{\alpha_n}(p_n)\dots a^\dagger_{\alpha_1}(p_1)|0\rangle$
4. Contract fields with creation operators via canonical (anti)commutation relations
5. Evaluate derivatives on plane waves: $\partial_\mu e^{-ipx} = -ip_\mu\,e^{-ipx}$
6. Remove external wavefunctions and the plane-wave factor
7. Multiply by $i$ to obtain the vertex factor: $V = i\,(\text{remaining coefficient})$


## Setup

In [1]:
import sys, os
print("Python:", sys.executable)

sys.path.insert(0, os.path.join(os.getcwd(), "code"))

from model_symbolica import (
    S, Expression, I, pi,
    delta, Delta, pcomp,
    plane_wave, contraction_rule,
    contract_to_full_expression,
    infer_derivative_targets,
    vertex_factor, simplify_deltas,
    compact_vertex_sum_form,
    compact_sum_notation,
)

from IPython.display import Markdown as md, display as dp

def show(expr):
    latex = expr.to_latex().strip().removeprefix("$$").removesuffix("$$")
    dp(md(f"$${latex}$$"))

print("model_symbolica loaded")

Python: /Users/rems/Library/CloudStorage/OneDrive-ETHZurich/ETHz_HS25/MScThesis/.venv/bin/python
model_symbolica loaded


## Step 1 -- Symbols and building blocks

The module defines function symbols:

| Symbol | Meaning |
|--------|---------|
| `phi(alpha, x)` | Scalar field operator |
| `adag(beta, p)` | Creation operator $a^\dagger_\beta(p)$ |
| `U(beta, p)` | External wavefunction (stripped later) |
| `delta(a, b)` | Kronecker delta $\delta_{ab}$ |
| `Delta(q)` | Momentum-conserving Dirac delta |
| `Dot(p, x)` | Minkowski dot product $p \cdot x$ |
| `pcomp(p, mu)` | Momentum component $p_\mu$ |

In [2]:
x = S('x')
p, alpha, beta = S('p', 'alpha', 'beta')

print("Plane wave exp(-i p.x):")
show(plane_wave(p, x))

print("Contraction rule [phi_alpha(x), a^dag_beta(p)]:")
show(contraction_rule(alpha, beta, p, x))

Plane wave exp(-i p.x):


$$\exp\!\left(-1𝑖 Dot\!\left(p,x\right)\right)$$

Contraction rule [phi_alpha(x), a^dag_beta(p)]:


$$\exp\!\left(-1𝑖 Dot\!\left(p,x\right)\right) U\!\left(beta,p\right) delta\!\left(alpha,beta\right)$$


## Step 2 -- Wick contractions (permutation sum)

$$
\langle 0|\phi_{\alpha_1}(x)\cdots\phi_{\alpha_n}(x)\,a^\dagger_{\beta_{\sigma(1)}}\cdots|0\rangle
= \sum_\sigma (\pm 1)^{\text{parity}(\sigma)} \prod_i \delta_{\alpha_i \beta_{\sigma(i)}}\, U_{\beta_{\sigma(i)}}(p_{\sigma(i)})\, e^{-i(\sum p) \cdot x}
$$

- Bosons: permanent (all signs $+1$)
- Fermions: determinant ($(-1)^{\text{parity}(\sigma)}$)

## Step 3 -- Derivative momentum factors

 $\partial_\mu \phi_k --> (-i)\, p_{k,\mu}$.


In [3]:
mu, nu = S('mu', 'nu')

deriv_idx, deriv_tgt = infer_derivative_targets([(0, [mu]), (1, [nu])])
print(f"indices = {deriv_idx}, targets = {deriv_tgt}")

indices = [mu, nu], targets = [0, 1]


## Step 4 -- Full vertex factor pipeline

`vertex_factor(...)` combines all steps:
1. Contract fields with creation operators
2. Multiply by coupling and derivative momentum factors
3. Integrate over $x$: $e^{-i(\sum p)\cdot x} \to (2\pi)^d \Delta(\sum p)$
4. Strip external wavefunctions $U(\beta, p) \to 1$
5. Multiply by $i$

## Examples

In [4]:
x = S('x')
d = S('d')
p1, p2, p3, p4, p5, p6 = S('p1', 'p2', 'p3', 'p4', 'p5', 'p6')
b1, b2, b3, b4, b5, b6 = S('b1', 'b2', 'b3', 'b4', 'b5', 'b6')
mu, nu = S('mu', 'nu')


def show_vertex(title, *, coupling, alphas, betas, ps,
                derivative_indices=(), derivative_targets=None,
                statistics="boson", species_map=None,
                compact_override=None, show_sum_notation=False):
    print("=" * 70)
    print(f"  {title}")

    expanded = vertex_factor(
        coupling=coupling, alphas=alphas, betas=betas, ps=ps, x=x,
        derivative_indices=derivative_indices,
        derivative_targets=derivative_targets,
        statistics=statistics,
        strip_externals=True, include_delta=True, d=d,
    )

    compact = compact_override
    if compact is None:
        compact = simplify_deltas(expanded, species_map=species_map)

    print("  Expanded:")
    print("   ", expanded)
    print("  Compact:")
    print("   ", compact)

    if show_sum_notation and derivative_indices:
        print("  Sum notation:")
        print("   ", compact_sum_notation(
            derivative_indices=derivative_indices,
            derivative_targets=derivative_targets,
            n_legs=len(ps),
        ))

    show(compact)
    return expanded, compact

### $\phi^4$: expect $V = 24\,i\,\lambda$

In [5]:
phi0 = S('phi0')
lam4 = S('lam4')

V_phi4, V_phi4_s = show_vertex(
    "lam4 * phi^4",
    coupling=lam4,
    alphas=[phi0]*4,
    betas=[b1, b2, b3, b4],
    ps=[p1, p2, p3, p4],
    species_map={b1: phi0, b2: phi0, b3: phi0, b4: phi0},
)

  lam4 * phi^4
  Expanded:
    24𝑖*lam4*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b2)*delta(phi0,b3)*delta(phi0,b4)*Delta(p1+p2+p3+p4)
  Compact:
    24𝑖*lam4*(2*𝜋)^d*Delta(p1+p2+p3+p4)


$$24𝑖 lam4 (2 \pi)^{d} Delta\!\left(p1+p2+p3+p4\right)$$

### $\phi^6$: expect $V = 720\,i\,\lambda_6$

In [6]:
lam6 = S('lam6')

V_phi6, V_phi6_s = show_vertex(
    "lam6 * phi^6",
    coupling=lam6,
    alphas=[phi0]*6,
    betas=[b1, b2, b3, b4, b5, b6],
    ps=[p1, p2, p3, p4, p5, p6],
    species_map={b1: phi0, b2: phi0, b3: phi0, b4: phi0, b5: phi0, b6: phi0},
)

  lam6 * phi^6
  Expanded:
    720𝑖*lam6*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b2)*delta(phi0,b3)*delta(phi0,b4)*delta(phi0,b5)*delta(phi0,b6)*Delta(p1+p2+p3+p4+p5+p6)
  Compact:
    720𝑖*lam6*(2*𝜋)^d*Delta(p1+p2+p3+p4+p5+p6)


$$720𝑖 lam6 (2 \pi)^{d} Delta\!\left(p1+p2+p3+p4+p5+p6\right)$$

### $\phi^2\chi^2$: expect $V = 4\,i\,g$

In [7]:
chi0 = S('chi0')
g_sym = S('g')

V_phi2chi2, V_phi2chi2_s = show_vertex(
    "g * phi^2 chi^2",
    coupling=g_sym,
    alphas=[phi0, phi0, chi0, chi0],
    betas=[b1, b2, b3, b4],
    ps=[p1, p2, p3, p4],
    species_map={b1: phi0, b2: phi0, b3: chi0, b4: chi0},
)

  g * phi^2 chi^2
  Expanded:
    1𝑖*g*(4*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b2)*delta(chi0,b3)*delta(chi0,b4)*Delta(p1+p2+p3+p4)+4*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b3)*delta(chi0,b2)*delta(chi0,b4)*Delta(p1+p2+p3+p4)+4*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b4)*delta(chi0,b2)*delta(chi0,b3)*Delta(p1+p2+p3+p4)+4*(2*𝜋)^d*delta(phi0,b2)*delta(phi0,b3)*delta(chi0,b1)*delta(chi0,b4)*Delta(p1+p2+p3+p4)+4*(2*𝜋)^d*delta(phi0,b2)*delta(phi0,b4)*delta(chi0,b1)*delta(chi0,b3)*Delta(p1+p2+p3+p4)+4*(2*𝜋)^d*delta(phi0,b3)*delta(phi0,b4)*delta(chi0,b1)*delta(chi0,b2)*Delta(p1+p2+p3+p4))
  Compact:
    4𝑖*g*(2*𝜋)^d*Delta(p1+p2+p3+p4)


$$4𝑖 g (2 \pi)^{d} Delta\!\left(p1+p2+p3+p4\right)$$

### Derivative interaction: $g_D\,(\partial_\mu\phi)(\partial_\nu\phi)\,\phi\,\phi$

In [8]:
gD = S('gD')
di, dt = infer_derivative_targets([(0, [mu]), (1, [nu])])

V_deriv_compact = compact_vertex_sum_form(
    coupling=gD,
    ps=[p1, p2, p3, p4],
    derivative_indices=di,
    derivative_targets=dt,
    d=d,
    field_species=[phi0, phi0, phi0, phi0],
    leg_species=[phi0, phi0, phi0, phi0],
)

V_deriv, V_deriv_s = show_vertex(
    "gD * (d_mu phi)(d_nu phi) phi phi",
    coupling=gD,
    alphas=[phi0]*4,
    betas=[b1, b2, b3, b4],
    ps=[p1, p2, p3, p4],
    derivative_indices=di,
    derivative_targets=dt,
    species_map={b1: phi0, b2: phi0, b3: phi0, b4: phi0},
    compact_override=V_deriv_compact,
    show_sum_notation=True,
)

  gD * (d_mu phi)(d_nu phi) phi phi
  Expanded:
    1𝑖*gD*(-2*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b2)*delta(phi0,b3)*delta(phi0,b4)*Delta(p1+p2+p3+p4)*pcomp(p1,mu)*pcomp(p2,nu)-2*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b2)*delta(phi0,b3)*delta(phi0,b4)*Delta(p1+p2+p3+p4)*pcomp(p1,mu)*pcomp(p3,nu)-2*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b2)*delta(phi0,b3)*delta(phi0,b4)*Delta(p1+p2+p3+p4)*pcomp(p1,mu)*pcomp(p4,nu)-2*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b2)*delta(phi0,b3)*delta(phi0,b4)*Delta(p1+p2+p3+p4)*pcomp(p1,nu)*pcomp(p2,mu)-2*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b2)*delta(phi0,b3)*delta(phi0,b4)*Delta(p1+p2+p3+p4)*pcomp(p1,nu)*pcomp(p3,mu)-2*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b2)*delta(phi0,b3)*delta(phi0,b4)*Delta(p1+p2+p3+p4)*pcomp(p1,nu)*pcomp(p4,mu)-2*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b2)*delta(phi0,b3)*delta(phi0,b4)*Delta(p1+p2+p3+p4)*pcomp(p2,mu)*pcomp(p3,nu)-2*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b2)*delta(phi0,b3)*delta(phi0,b4)*Delta(p1+p2+p3+p4)*pcomp(p2,mu)*pcomp(p4,nu)-2*(2*𝜋)^d*delta(phi0

$$-1𝑖 gD (2 \pi)^{d} \left(2 pcomp\!\left(p1,mu\right) pcomp\!\left(p2,nu\right)+2 pcomp\!\left(p1,mu\right) pcomp\!\left(p3,nu\right)+2 pcomp\!\left(p1,mu\right) pcomp\!\left(p4,nu\right)+2 pcomp\!\left(p1,nu\right) pcomp\!\left(p2,mu\right)+2 pcomp\!\left(p1,nu\right) pcomp\!\left(p3,mu\right)+2 pcomp\!\left(p1,nu\right) pcomp\!\left(p4,mu\right)+2 pcomp\!\left(p2,mu\right) pcomp\!\left(p3,nu\right)+2 pcomp\!\left(p2,mu\right) pcomp\!\left(p4,nu\right)+2 pcomp\!\left(p2,nu\right) pcomp\!\left(p3,mu\right)+2 pcomp\!\left(p2,nu\right) pcomp\!\left(p4,mu\right)+2 pcomp\!\left(p3,mu\right) pcomp\!\left(p4,nu\right)+2 pcomp\!\left(p3,nu\right) pcomp\!\left(p4,mu\right)\right) Delta\!\left(p1+p2+p3+p4\right)$$

### Derivative interaction (same index): $g_{D2}\,(\partial_\mu\phi)(\partial_\mu\phi)\,\phi\,\phi$

In [9]:
gD2 = S('gD2')
di2, dt2 = infer_derivative_targets([(0, [mu]), (1, [mu])])

V_deriv2_compact = compact_vertex_sum_form(
    coupling=gD2,
    ps=[p1, p2, p3, p4],
    derivative_indices=di2,
    derivative_targets=dt2,
    d=d,
    field_species=[phi0, phi0, phi0, phi0],
    leg_species=[phi0, phi0, phi0, phi0],
)

V_deriv2, V_deriv2_s = show_vertex(
    "gD2 * (d_mu phi)(d_mu phi) phi phi",
    coupling=gD2,
    alphas=[phi0]*4,
    betas=[b1, b2, b3, b4],
    ps=[p1, p2, p3, p4],
    derivative_indices=di2,
    derivative_targets=dt2,
    species_map={b1: phi0, b2: phi0, b3: phi0, b4: phi0},
    compact_override=V_deriv2_compact,
    show_sum_notation=True,
)

  gD2 * (d_mu phi)(d_mu phi) phi phi
  Expanded:
    1𝑖*gD2*(-4*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b2)*delta(phi0,b3)*delta(phi0,b4)*Delta(p1+p2+p3+p4)*pcomp(p1,mu)*pcomp(p2,mu)-4*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b2)*delta(phi0,b3)*delta(phi0,b4)*Delta(p1+p2+p3+p4)*pcomp(p1,mu)*pcomp(p3,mu)-4*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b2)*delta(phi0,b3)*delta(phi0,b4)*Delta(p1+p2+p3+p4)*pcomp(p1,mu)*pcomp(p4,mu)-4*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b2)*delta(phi0,b3)*delta(phi0,b4)*Delta(p1+p2+p3+p4)*pcomp(p2,mu)*pcomp(p3,mu)-4*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b2)*delta(phi0,b3)*delta(phi0,b4)*Delta(p1+p2+p3+p4)*pcomp(p2,mu)*pcomp(p4,mu)-4*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b2)*delta(phi0,b3)*delta(phi0,b4)*Delta(p1+p2+p3+p4)*pcomp(p3,mu)*pcomp(p4,mu))
  Compact:
    -1𝑖*gD2*(2*𝜋)^d*(4*pcomp(p1,mu)*pcomp(p2,mu)+4*pcomp(p1,mu)*pcomp(p3,mu)+4*pcomp(p1,mu)*pcomp(p4,mu)+4*pcomp(p2,mu)*pcomp(p3,mu)+4*pcomp(p2,mu)*pcomp(p4,mu)+4*pcomp(p3,mu)*pcomp(p4,mu))*Delta(p1+p2+p3+p4)
  Sum notation:
    (2)! * 

$$-1𝑖 gD2 (2 \pi)^{d} \left(4 pcomp\!\left(p1,mu\right) pcomp\!\left(p2,mu\right)+4 pcomp\!\left(p1,mu\right) pcomp\!\left(p3,mu\right)+4 pcomp\!\left(p1,mu\right) pcomp\!\left(p4,mu\right)+4 pcomp\!\left(p2,mu\right) pcomp\!\left(p3,mu\right)+4 pcomp\!\left(p2,mu\right) pcomp\!\left(p4,mu\right)+4 pcomp\!\left(p3,mu\right) pcomp\!\left(p4,mu\right)\right) Delta\!\left(p1+p2+p3+p4\right)$$

### Multi-species: $g_{ijk}\,\phi_i^2\,\phi_j^2\,\phi_k^2$

In [10]:
idx_i, idx_j, idx_k = S('i', 'j', 'k')
gijk = S('gijk')

V_multi, V_multi_s = show_vertex(
    "gijk(i,j,k) * phi_i^2 phi_j^2 phi_k^2",
    coupling=gijk(idx_i, idx_j, idx_k),
    alphas=[idx_i, idx_i, idx_j, idx_j, idx_k, idx_k],
    betas=[b1, b2, b3, b4, b5, b6],
    ps=[p1, p2, p3, p4, p5, p6],
)

  gijk(i,j,k) * phi_i^2 phi_j^2 phi_k^2
  Expanded:
    1𝑖*(8*(2*𝜋)^d*delta(i,b1)*delta(i,b2)*delta(j,b3)*delta(j,b4)*delta(k,b5)*delta(k,b6)*Delta(p1+p2+p3+p4+p5+p6)+8*(2*𝜋)^d*delta(i,b1)*delta(i,b2)*delta(j,b3)*delta(j,b5)*delta(k,b4)*delta(k,b6)*Delta(p1+p2+p3+p4+p5+p6)+8*(2*𝜋)^d*delta(i,b1)*delta(i,b2)*delta(j,b3)*delta(j,b6)*delta(k,b4)*delta(k,b5)*Delta(p1+p2+p3+p4+p5+p6)+8*(2*𝜋)^d*delta(i,b1)*delta(i,b2)*delta(j,b4)*delta(j,b5)*delta(k,b3)*delta(k,b6)*Delta(p1+p2+p3+p4+p5+p6)+8*(2*𝜋)^d*delta(i,b1)*delta(i,b2)*delta(j,b4)*delta(j,b6)*delta(k,b3)*delta(k,b5)*Delta(p1+p2+p3+p4+p5+p6)+8*(2*𝜋)^d*delta(i,b1)*delta(i,b2)*delta(j,b5)*delta(j,b6)*delta(k,b3)*delta(k,b4)*Delta(p1+p2+p3+p4+p5+p6)+8*(2*𝜋)^d*delta(i,b1)*delta(i,b3)*delta(j,b2)*delta(j,b4)*delta(k,b5)*delta(k,b6)*Delta(p1+p2+p3+p4+p5+p6)+8*(2*𝜋)^d*delta(i,b1)*delta(i,b3)*delta(j,b2)*delta(j,b5)*delta(k,b4)*delta(k,b6)*Delta(p1+p2+p3+p4+p5+p6)+8*(2*𝜋)^d*delta(i,b1)*delta(i,b3)*delta(j,b2)*delta(j,b6)*delta(k,b4)*delta(k,b5)*De

$$1𝑖 \left(8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b2\right) delta\!\left(j,b3\right) delta\!\left(j,b4\right) delta\!\left(k,b5\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b2\right) delta\!\left(j,b3\right) delta\!\left(j,b5\right) delta\!\left(k,b4\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b2\right) delta\!\left(j,b3\right) delta\!\left(j,b6\right) delta\!\left(k,b4\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b2\right) delta\!\left(j,b4\right) delta\!\left(j,b5\right) delta\!\left(k,b3\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b2\right) delta\!\left(j,b4\right) delta\!\left(j,b6\right) delta\!\left(k,b3\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b2\right) delta\!\left(j,b5\right) delta\!\left(j,b6\right) delta\!\left(k,b3\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b3\right) delta\!\left(j,b2\right) delta\!\left(j,b4\right) delta\!\left(k,b5\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b3\right) delta\!\left(j,b2\right) delta\!\left(j,b5\right) delta\!\left(k,b4\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b3\right) delta\!\left(j,b2\right) delta\!\left(j,b6\right) delta\!\left(k,b4\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b3\right) delta\!\left(j,b4\right) delta\!\left(j,b5\right) delta\!\left(k,b2\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b3\right) delta\!\left(j,b4\right) delta\!\left(j,b6\right) delta\!\left(k,b2\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b3\right) delta\!\left(j,b5\right) delta\!\left(j,b6\right) delta\!\left(k,b2\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b4\right) delta\!\left(j,b2\right) delta\!\left(j,b3\right) delta\!\left(k,b5\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b4\right) delta\!\left(j,b2\right) delta\!\left(j,b5\right) delta\!\left(k,b3\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b4\right) delta\!\left(j,b2\right) delta\!\left(j,b6\right) delta\!\left(k,b3\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b4\right) delta\!\left(j,b3\right) delta\!\left(j,b5\right) delta\!\left(k,b2\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b4\right) delta\!\left(j,b3\right) delta\!\left(j,b6\right) delta\!\left(k,b2\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b4\right) delta\!\left(j,b5\right) delta\!\left(j,b6\right) delta\!\left(k,b2\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b5\right) delta\!\left(j,b2\right) delta\!\left(j,b3\right) delta\!\left(k,b4\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b5\right) delta\!\left(j,b2\right) delta\!\left(j,b4\right) delta\!\left(k,b3\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b5\right) delta\!\left(j,b2\right) delta\!\left(j,b6\right) delta\!\left(k,b3\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b5\right) delta\!\left(j,b3\right) delta\!\left(j,b4\right) delta\!\left(k,b2\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b5\right) delta\!\left(j,b3\right) delta\!\left(j,b6\right) delta\!\left(k,b2\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b5\right) delta\!\left(j,b4\right) delta\!\left(j,b6\right) delta\!\left(k,b2\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b6\right) delta\!\left(j,b2\right) delta\!\left(j,b3\right) delta\!\left(k,b4\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b6\right) delta\!\left(j,b2\right) delta\!\left(j,b4\right) delta\!\left(k,b3\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b6\right) delta\!\left(j,b2\right) delta\!\left(j,b5\right) delta\!\left(k,b3\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b6\right) delta\!\left(j,b3\right) delta\!\left(j,b4\right) delta\!\left(k,b2\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b6\right) delta\!\left(j,b3\right) delta\!\left(j,b5\right) delta\!\left(k,b2\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b6\right) delta\!\left(j,b4\right) delta\!\left(j,b5\right) delta\!\left(k,b2\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b3\right) delta\!\left(j,b1\right) delta\!\left(j,b4\right) delta\!\left(k,b5\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b3\right) delta\!\left(j,b1\right) delta\!\left(j,b5\right) delta\!\left(k,b4\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b3\right) delta\!\left(j,b1\right) delta\!\left(j,b6\right) delta\!\left(k,b4\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b3\right) delta\!\left(j,b4\right) delta\!\left(j,b5\right) delta\!\left(k,b1\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b3\right) delta\!\left(j,b4\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b3\right) delta\!\left(j,b5\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b4\right) delta\!\left(j,b1\right) delta\!\left(j,b3\right) delta\!\left(k,b5\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b4\right) delta\!\left(j,b1\right) delta\!\left(j,b5\right) delta\!\left(k,b3\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b4\right) delta\!\left(j,b1\right) delta\!\left(j,b6\right) delta\!\left(k,b3\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b4\right) delta\!\left(j,b3\right) delta\!\left(j,b5\right) delta\!\left(k,b1\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b4\right) delta\!\left(j,b3\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b4\right) delta\!\left(j,b5\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b5\right) delta\!\left(j,b1\right) delta\!\left(j,b3\right) delta\!\left(k,b4\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b5\right) delta\!\left(j,b1\right) delta\!\left(j,b4\right) delta\!\left(k,b3\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b5\right) delta\!\left(j,b1\right) delta\!\left(j,b6\right) delta\!\left(k,b3\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b5\right) delta\!\left(j,b3\right) delta\!\left(j,b4\right) delta\!\left(k,b1\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b5\right) delta\!\left(j,b3\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b5\right) delta\!\left(j,b4\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b3\right) delta\!\left(k,b4\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b4\right) delta\!\left(k,b3\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b5\right) delta\!\left(k,b3\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b6\right) delta\!\left(j,b3\right) delta\!\left(j,b4\right) delta\!\left(k,b1\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b6\right) delta\!\left(j,b3\right) delta\!\left(j,b5\right) delta\!\left(k,b1\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b6\right) delta\!\left(j,b4\right) delta\!\left(j,b5\right) delta\!\left(k,b1\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b4\right) delta\!\left(j,b1\right) delta\!\left(j,b2\right) delta\!\left(k,b5\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b4\right) delta\!\left(j,b1\right) delta\!\left(j,b5\right) delta\!\left(k,b2\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b4\right) delta\!\left(j,b1\right) delta\!\left(j,b6\right) delta\!\left(k,b2\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b4\right) delta\!\left(j,b2\right) delta\!\left(j,b5\right) delta\!\left(k,b1\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b4\right) delta\!\left(j,b2\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b4\right) delta\!\left(j,b5\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b2\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b5\right) delta\!\left(j,b1\right) delta\!\left(j,b2\right) delta\!\left(k,b4\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b5\right) delta\!\left(j,b1\right) delta\!\left(j,b4\right) delta\!\left(k,b2\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b5\right) delta\!\left(j,b1\right) delta\!\left(j,b6\right) delta\!\left(k,b2\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b5\right) delta\!\left(j,b2\right) delta\!\left(j,b4\right) delta\!\left(k,b1\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b5\right) delta\!\left(j,b2\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b5\right) delta\!\left(j,b4\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b2\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b2\right) delta\!\left(k,b4\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b4\right) delta\!\left(k,b2\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b5\right) delta\!\left(k,b2\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b6\right) delta\!\left(j,b2\right) delta\!\left(j,b4\right) delta\!\left(k,b1\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b6\right) delta\!\left(j,b2\right) delta\!\left(j,b5\right) delta\!\left(k,b1\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b6\right) delta\!\left(j,b4\right) delta\!\left(j,b5\right) delta\!\left(k,b1\right) delta\!\left(k,b2\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b5\right) delta\!\left(j,b1\right) delta\!\left(j,b2\right) delta\!\left(k,b3\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b5\right) delta\!\left(j,b1\right) delta\!\left(j,b3\right) delta\!\left(k,b2\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b5\right) delta\!\left(j,b1\right) delta\!\left(j,b6\right) delta\!\left(k,b2\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b5\right) delta\!\left(j,b2\right) delta\!\left(j,b3\right) delta\!\left(k,b1\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b5\right) delta\!\left(j,b2\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b5\right) delta\!\left(j,b3\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b2\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b2\right) delta\!\left(k,b3\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b3\right) delta\!\left(k,b2\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b5\right) delta\!\left(k,b2\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b6\right) delta\!\left(j,b2\right) delta\!\left(j,b3\right) delta\!\left(k,b1\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b6\right) delta\!\left(j,b2\right) delta\!\left(j,b5\right) delta\!\left(k,b1\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b6\right) delta\!\left(j,b3\right) delta\!\left(j,b5\right) delta\!\left(k,b1\right) delta\!\left(k,b2\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b5\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b2\right) delta\!\left(k,b3\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b5\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b3\right) delta\!\left(k,b2\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b5\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b4\right) delta\!\left(k,b2\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b5\right) delta\!\left(i,b6\right) delta\!\left(j,b2\right) delta\!\left(j,b3\right) delta\!\left(k,b1\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b5\right) delta\!\left(i,b6\right) delta\!\left(j,b2\right) delta\!\left(j,b4\right) delta\!\left(k,b1\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b5\right) delta\!\left(i,b6\right) delta\!\left(j,b3\right) delta\!\left(j,b4\right) delta\!\left(k,b1\right) delta\!\left(k,b2\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)\right) gijk\!\left(i,j,k\right)$$

In [ ]:
import model_symbolica as ms

# ---------------------------------------------------------------------------
# Symbols
# ---------------------------------------------------------------------------
psibar0, psi0 = S('psibar0', 'psi0')

yF = S('yF')
g_psi4 = S('g_psi4')

# Spinor indices / external polarizations
i_psi_bar, i_psi = S('i_psi_bar', 'i_psi')
s1, s2, s3, s4 = S('s1', 's2', 's3', 's4')

alpha_s, beta_s = S('alpha_s', 'beta_s')
i1, i2, i3, i4 = S('i1', 'i2', 'i3', 'i4')

# ---------------------------------------------------------------------------
# 1) Yukawa-like toy term: L_int = yF * psibar * psi * phi
# ---------------------------------------------------------------------------
L_yukawa = dict(
    coupling=yF,
    alphas=[psibar0, psi0, phi0],
    betas=[b1, b2, b3],
    ps=[p1, p2, p3],
    statistics='fermion',
    field_roles=['psibar', 'psi', 'scalar'],
    leg_roles=['psibar', 'psi', 'scalar'],
    field_spinor_indices=[i_psi_bar, i_psi, None],
    leg_spins=[s1, s2, s3],
)

V_yuk = vertex_factor(**L_yukawa, x=x, d=d)
V_yuk_c = simplify_deltas(V_yuk, species_map={b1: psibar0, b2: psi0, b3: phi0})
expected_yuk = I * yF * (2 * pi) ** d * Delta(p1 + p2 + p3)

assert (
    V_yuk_c.expand().to_canonical_string()
    == expected_yuk.expand().to_canonical_string()
), f'Yukawa-like fermion vertex failed: got {V_yuk_c}, expected {expected_yuk}'
print('Fermion Yukawa-like vertex: PASS')

# Unstripped version should contain UF/UbarF wavefunctions
V_yuk_full = vertex_factor(**L_yukawa, x=x, d=d, strip_externals=False)
V_yuk_full_c = simplify_deltas(V_yuk_full, species_map={b1: psibar0, b2: psi0, b3: phi0})
V_yuk_full_str = V_yuk_full_c.to_canonical_string()
assert 'UF' in V_yuk_full_str and 'UbarF' in V_yuk_full_str
print('Fermion Yukawa unstripped: PASS (contains UF/UbarF)')

# ---------------------------------------------------------------------------
# 2) Four-fermion bilinear: L_int = -(g/2) (psibar psi)(psibar psi)
#    Spinor-delta mode via leg_spinor_indices
# ---------------------------------------------------------------------------
L_psibar_psi_sq_spinor = dict(
    coupling=-g_psi4 / Expression.num(2),
    alphas=[psibar0, psi0, psibar0, psi0],
    betas=[b1, b2, b3, b4],
    ps=[p1, p2, p3, p4],
    statistics='fermion',
    field_roles=['psibar', 'psi', 'psibar', 'psi'],
    leg_roles=['psibar', 'psi', 'psibar', 'psi'],
    field_spinor_indices=[alpha_s, alpha_s, beta_s, beta_s],
    leg_spins=[s1, s2, s3, s4],
    leg_spinor_indices=[i1, i2, i3, i4],
)

V_spin = vertex_factor(**L_psibar_psi_sq_spinor, x=x, d=d)
V_spin_c = simplify_deltas(V_spin, species_map={b1: psibar0, b2: psi0, b3: psibar0, b4: psi0})

expected_spin = (
    -I * g_psi4 * (2 * pi) ** d * Delta(p1 + p2 + p3 + p4)
    * (
        ms.delta_s(i1, i2) * ms.delta_s(i3, i4)
        - ms.delta_s(i1, i4) * ms.delta_s(i3, i2)
    )
)

assert (
    V_spin_c.expand().to_canonical_string()
    == expected_spin.expand().to_canonical_string()
), f'Four-fermion spinor-delta failed: got {V_spin_c}, expected {expected_spin}'
print('Four-fermion (psibar psi)^2 spinor-delta: PASS')
